# **Main Plot for Resolution**

Glasses Cams ────> HEVC Stream ──[BT-Classic/WiFi Direct]──> Phone Decoder ──> Cropped and upscaled Stream --> Preview UI --> Encoder --> Stream


Phone Mic ──> H.264/AAC ──> Back to Glasses 

In [3]:
import re
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter, defaultdict
import os
from datetime import datetime

# Returns lists of (timestamp, resolution) to preserve frequencies and times
def parse_stream_resolutions(log_file):
    """
    Parses a log file to extract and differentiate resolutions for:
    1. Glasses Streamed to Phone (G2P - from CCodecConfig c2 config diff blocks with raw.size for HEVC only)
    2. Phone Streamed to IG/App (P2S - from 'scaler_node.cc' or 'cam[56]_eaf' logs)

    Returns: two lists of (timestamp, resolution) in "WxH" format (g2p_list, p2s_list).
    """
    def parse_timestamp(line):
        match = re.match(r'(\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d{3})', line)
        if match:
            try:
                return datetime.strptime(match.group(1), '%m-%d %H:%M:%S.%f')
            except ValueError:
                return None
        return None

    try:
        with open(log_file, 'r', encoding='utf-8') as f:
            lines = f.readlines()
    except FileNotFoundError:
        print(f"Error: File not found at {log_file}")
        return [], []
    except Exception as e:
        print(f"Error reading file {log_file}: {e}")
        return [], []

    # 1. Glasses to Phone (G2P) - Parse from CCodecConfig c2 config diff blocks, only for HEVC
    g2p_matches = []
    i = 0
    while i < len(lines):
        line = lines[i]
        if re.search(r'c2 config diff is', line, re.IGNORECASE):
            ts = parse_timestamp(line)
            if ts is None:
                i += 1
                continue
            block = line
            i += 1
            while i < len(lines) and re.search(r'D CCodecConfig:\s*c2::u32', lines[i], re.IGNORECASE):
                block += '\n' + lines[i]
                i += 1
            # Skip if this is not a HEVC block: for full configs, check mime; for updates (no mime), proceed
            if 'input.media-type.value' in block:
                if 'video/hevc' not in block:
                    continue
            # Extract raw.size.width and height
            match_w = re.search(r'raw\.size\.width = (\d+)', block)
            match_h = re.search(r'raw\.size\.height = (\d+)', block)
            if match_w and match_h:
                w = match_w.group(1)
                h = match_h.group(1)
                res = f"{min(int(w), int(h))}x{max(int(w), int(h))}"
                g2p_matches.append((ts, res))
        else:
            i += 1

    # 2. Phone to Stream (P2S)
    p2s_matches = []
    for line in lines:
        match = re.search(
            r'(?:scaler_node\.cc.*?resolution:\s*(?P<w1>\d+)x(?P<h1>\d+))|'  # Pattern 1: scaler_node
            r'(?:cam[2356]_eaf.*?res\s*(?P<w2>\d+)x(?P<h2>\d+))',               # Pattern 2: cam2, 3, 5, or 6 eaf
            line, re.IGNORECASE)
        
        if match:
            ts = parse_timestamp(line)
            if ts is None:
                continue
            
            w, h = None, None
            # Check which named group actually matched
            if match.group('w1') and match.group('h1'):
                w = match.group('w1')
                h = match.group('h1')
            elif match.group('w2') and match.group('h2'):
                w = match.group('w2')
                h = match.group('h2')
            else:
                continue # No valid group was captured

            # Normalize resolution string to min_dim x max_dim
            res = f"{min(int(w), int(h))}x{max(int(w), int(h))}"
            p2s_matches.append((ts, res))

    return g2p_matches, p2s_matches


def create_resolution_maps(all_resolutions):

    unique_resolutions = set(all_resolutions)

    if not unique_resolutions:
        return {}, {}

    width_groups = defaultdict(list)

    for res in unique_resolutions:

        try:
            width, height = map(int, res.split('x'))
            if width == 2448 and height == 3440:
                continue

            if height not in width_groups[width]:
                width_groups[width].append(height)
        except ValueError:
            continue

    sorted_widths = sorted(width_groups.keys())

    forward_map = {}
    reverse_map = {}
    mapping_id = 1

    for width in sorted_widths:
        heights = sorted(width_groups[width])
        group_key = f"{width}x{heights}"

        forward_map[mapping_id] = group_key
        for height in heights:
            reverse_map[f"{width}x{height}"] = mapping_id
        mapping_id += 1

    return forward_map, reverse_map


def compute_mapping_id_frequencies(resolutions_list, reverse_mapping):
    """
    Compute frequency counts per mapping ID.
    'resolutions_list' is expected to be a list, potentially with duplicates.
    """
    res_counter = Counter(resolutions_list)
    id_frequencies = defaultdict(int)
    excluded_res = "2448x3440"

    for res, count in res_counter.items():
        if res == excluded_res:
            continue
        map_id = reverse_mapping.get(res, None)
        if map_id is not None:
            id_frequencies[map_id] += count

    return dict(id_frequencies)

def compute_durations(ts, ids, min_ts, max_ts):
    if not ts:
        return {}

    t_rel = [((t - min_ts).total_seconds() * 1000) for t in ts]
    max_t_rel = ((max_ts - min_ts).total_seconds() * 1000)

    if t_rel[0] > 0:
        t_rel = [0.0] + t_rel
        ids = [ids[0]] + ids

    if t_rel[-1] < max_t_rel:
        t_rel = t_rel + [max_t_rel]
        ids = ids + [ids[-1]]

    durations = defaultdict(float)
    for i in range(len(t_rel) - 1):
        dur = t_rel[i+1] - t_rel[i]
        durations[ids[i]] += dur

    return durations

def plot_cdf(g2p_list, p2s_list, save_label, reverse_map, save_path=None):
    g2p_valid = [(ts, res) for ts, res in g2p_list if res in reverse_map]
    if g2p_valid:
        g2p_valid.sort(key=lambda x: x[0])
        g2p_ts = [item[0] for item in g2p_valid]
        g2p_ids = [reverse_map[res] for res in [item[1] for item in g2p_valid]]
    else:
        g2p_ts = []
        g2p_ids = []

    # Filter and sort P2S
    p2s_valid = [(ts, res) for ts, res in p2s_list if res in reverse_map]
    if p2s_valid:
        p2s_valid.sort(key=lambda x: x[0])
        p2s_ts = [item[0] for item in p2s_valid]
        p2s_ids = [reverse_map[res] for res in [item[1] for item in p2s_valid]]
    else:
        p2s_ts = []
        p2s_ids = []

    all_ts = g2p_ts + p2s_ts
    if not all_ts:
        return

    min_ts = min(all_ts)
    max_ts = max(all_ts)

    g2p_durs = compute_durations(g2p_ts, g2p_ids, min_ts, max_ts)
    p2s_durs = compute_durations(p2s_ts, p2s_ids, min_ts, max_ts)

    max_id = max(reverse_map.values()) if reverse_map else 0

    g_cum = None
    if g2p_durs:
        g_times = [g2p_durs.get(i, 0.0) for i in range(1, max_id + 1)]
        g_total = sum(g_times)
        if g_total > 0:
            g_cum = np.cumsum(g_times) / g_total

    p_cum = None
    if p2s_durs:
        p_times = [p2s_durs.get(i, 0.0) for i in range(1, max_id + 1)]
        p_total = sum(p_times)
        if p_total > 0:
            p_cum = np.cumsum(p_times) / p_total

    if g_cum is None and p_cum is None:
        return

    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(1, max_id + 1)
    plotted = False

    if g_cum is not None:
        ax.step(x, g_cum, where='post', label='G2P', linewidth=4)
        plotted = True

    if p_cum is not None:
        ax.step(x, p_cum, where='post', label='P2S', linewidth=4, linestyle='--')
        plotted = True

    if not plotted:
        plt.close()
        return
    else:
        ax.set_xlabel('Resolution', fontsize=55)
        ax.set_ylabel('CDF', fontsize=55)
        ax.set_ylim(0, 1.05)
        ax.tick_params(axis='both', labelsize=55)
        ax.set_xticks(x)
        ax.grid(True, alpha=0.7)
        ax.legend(fontsize=40, loc='upper left')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()

def main():
    """
    Main function to process files, create mappings, and plot resolution ID changes over time.
    """
    log_files = [
        "Test01_logcat_WD_RB_insta_Live.txt",
        "Test01_logcat_BT_RB_insta_Live.txt",
        "Test01_logcat_BT_RB_FB_Live.txt",
        "Test01_logcat_WD_RB_FB_Live.txt",
        "Test01_logcat_BT_RB_insta_Live_Distance_8.txt",
        "Test01_logcat_WD_RB_insta_Live_Distance_8m.txt",
        "Test01_logcat_WD_RB_insta_Live_Distance_20.txt",
        "Test01_logcat_BT_RB_insta_Live_Distance_20.txt",
        "Test01_logcat_WD_RB_insta_Live_Distance_30.txt",
        "Test01_logcat_BT_RB_insta_Live_Distance_30.txt"
    ]

    file_results = {}
    all_resolutions_with_duplicates = []
    excluded_res = "2448x3440"

    for log_file in log_files:
        # Simple check to skip missing files
        if not os.path.exists(log_file):
            print(f"Skipping missing file: {log_file}")
            continue
            
        print(f"Parsing {log_file}...")
        g2p_list, p2s_list = parse_stream_resolutions(log_file)

        file_results[log_file] = {
            'g2p': g2p_list,
            'p2s': p2s_list
        }

        all_resolutions_with_duplicates.extend([res for _, res in g2p_list])
        all_resolutions_with_duplicates.extend([res for _, res in p2s_list])

    print("All files parsed.")

    mapping, reverse_mapping = create_resolution_maps(set(all_resolutions_with_duplicates))

    print("--- Global Resolution Mapping ---")
    print("ID is assigned based on increasing width.\n")
    if not mapping:
        print("No resolution data found to create a map.")
    else:
        # Print mapping with ID first for clarity with the plot
        for map_id, res_group in sorted(mapping.items()):
            print(f"ID: {map_id} -> Group: {res_group}")
    print("\n" + "="*40 + "\n")

    # Check if any files were actually processed
    if not file_results:
        print("No log files were found or processed. Exiting.")
        return

    for log_file in file_results.keys(): # Iterate over processed files
        parts = log_file.split('_')
        connection = 'WD' if 'WD' in parts else ('BT' if 'BT' in parts else '_')
        app_match = re.search(r'RB_(.*?)\.txt', log_file)
        app = app_match.group(1) if app_match else 'unknown'

        print(f"--- Detailed Results for {app.capitalize()} ({connection}): {log_file} ---")

        g2p_results_list = file_results[log_file]['g2p']
        p2s_results_list = file_results[log_file]['p2s']

        g2p_res_only = [res for _, res in g2p_results_list]
        p2s_res_only = [res for _, res in p2s_results_list]

        if g2p_res_only:
            g2p_counts = Counter(r for r in g2p_res_only)
            if g2p_counts:
                print("\n  Glasses to Phone (G2P) Resolutions (from CCodecConfig):")
                for resolution, count in sorted(g2p_counts.items()):
                    map_id = reverse_mapping.get(resolution, "N/A (Excluded or Not Found)")
                    print(f" - Resolution: {resolution}, Frequency: {count}, Mapping ID: {map_id}")
            else:
                print("\n  No non-excluded Glasses to Phone (G2P) Resolutions found.")
        else:
            print("\n  No Glasses to Phone (G2P) Resolutions found in this file.")

        if p2s_res_only:
            p2s_counts = Counter(r for r in p2s_res_only)
            if p2s_counts:
                print(f"\n  Phone to Stream (P2S) Resolutions (from scaler_node/cam_eaf):")
                for resolution, count in sorted(p2s_counts.items()):
                    map_id = reverse_mapping.get(resolution, "N/A (Excluded or Not Found)")
                    print(f" - Resolution: {resolution}, Frequency: {count}, Mapping ID: {map_id}")
            else:
                print("\n  No non-excluded Phone to Stream (P2S) Resolutions found.")
        else:
            print(f"\n  No Phone to Stream (P2S) Resolutions found in this file.")

        print("\n" + "="*40 + "\n")

    print("Generating plots...")
    os.makedirs("./Figures", exist_ok=True)
    for log_file in file_results.keys(): # Iterate over processed files
        parts = log_file.split('_')
        connection = 'WD' if 'WD' in parts else ('BT' if 'BT' in parts else '_')
        app_match = re.search(r'RB_(.*?)\.txt', log_file)
        app = app_match.group(1) if app_match else 'unknown'

        g2p_list = file_results[log_file]['g2p']
        p2s_list = file_results[log_file]['p2s']

        save_label = f"{connection}_{app}"

        timeline_save_path = None
        cdf_save_path = f"../Plots/{save_label}_cdf.png"
        
        plot_cdf(g2p_list, p2s_list, save_label, reverse_mapping, save_path=cdf_save_path)

    print("Done.")

if __name__ == "__main__":
    main()

Parsing Test01_logcat_WD_RB_insta_Live.txt...
Parsing Test01_logcat_BT_RB_insta_Live.txt...
Parsing Test01_logcat_BT_RB_FB_Live.txt...
Parsing Test01_logcat_WD_RB_FB_Live.txt...
Parsing Test01_logcat_BT_RB_insta_Live_Distance_8.txt...
Parsing Test01_logcat_WD_RB_insta_Live_Distance_8m.txt...
Parsing Test01_logcat_WD_RB_insta_Live_Distance_20.txt...
Parsing Test01_logcat_BT_RB_insta_Live_Distance_20.txt...
Parsing Test01_logcat_WD_RB_insta_Live_Distance_30.txt...
Parsing Test01_logcat_BT_RB_insta_Live_Distance_30.txt...
All files parsed.
--- Global Resolution Mapping ---
ID is assigned based on increasing width.

ID: 1 -> Group: 360x[640]
ID: 2 -> Group: 432x[768]
ID: 3 -> Group: 504x[896]
ID: 4 -> Group: 720x[1280]
ID: 5 -> Group: 1080x[1920]
ID: 6 -> Group: 1836x[3264]
ID: 7 -> Group: 2268x[4032]


--- Detailed Results for Insta_live (WD): Test01_logcat_WD_RB_insta_Live.txt ---

  Glasses to Phone (G2P) Resolutions (from CCodecConfig):
 - Resolution: 504x896, Frequency: 7, Mapping ID: